# 03 - MMASH preprocessing

Pre-sleep (prospective) window, HRV with gap-aware successive differences, actigraphy and diary features, questionnaire cleaning. Code: `src/preprocessing/mmash_preprocess.py`.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from IPython.display import Image, display, Markdown
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200)
M = ROOT / 'results' / 'metrics'; F = ROOT / 'results' / 'figures'; P = ROOT / 'data' / 'processed'
from src.preprocessing.mmash_preprocess import build_mmash, MMASH_MODALITIES, TARGETS
out = build_mmash(save=False)

## QC log - every inclusion decision

In [2]:
out['qc']

,user,load_issues,pre_sleep_hours,rr_beats_pre_sleep,rr_pct_artefact_pre_sleep,hrv_5min_segments_kept,actigraph_worn_hours_pre_sleep,diary_code0_rows_excluded,diary_rows_without_end_excluded,questionnaire_values_set_missing,included,reason
0,1,none,14.60,61474.0,2.61,174.0,11.51,1.0,1.0,none,True,passed
1,2,none,14.78,51733.0,4.45,161.0,11.05,2.0,1.0,none,True,passed
2,3,none,13.23,52854.0,2.68,154.0,10.65,1.0,0.0,none,True,passed
3,4,none,15.82,45157.0,24.39,15.0,12.08,1.0,0.0,none,True,passed
4,5,none,14.51,53031.0,2.72,172.0,10.83,3.0,0.0,none,True,passed
5,6,none,16.14,66471.0,2.42,186.0,11.28,1.0,0.0,none,True,passed
6,7,none,14.59,56566.0,0.13,174.0,11.02,1.0,0.0,STAI2=0,True,passed
7,8,user_8/RR: day codes [-29] recoded to 2 (23434...,14.40,48931.0,1.88,159.0,10.02,3.0,0.0,none,True,passed
8,9,user_9/RR: day codes [-29] recoded to 2 (24165...,14.39,52804.0,0.45,164.0,10.38,3.0,0.0,none,True,passed
9,10,none,17.58,59619.0,3.61,197.0,13.19,2.0,0.0,none,True,passed


## Participant-level features by modality

In [3]:
for m, cols in MMASH_MODALITIES.items():
    display(Markdown(f'### {m}'))
    display(out['features'][['user'] + cols].round(2))

### Physiological

,user,phys_hr_mean,phys_rmssd,phys_sdnn,phys_pnn50,phys_hr_evening
0,1,89.75,27.85,61.72,6.51,89.89
1,2,78.13,27.11,66.43,5.80,69.14
2,3,85.08,39.48,61.36,14.00,82.18
3,4,75.78,46.82,90.69,22.11,69.56
4,5,78.98,33.64,71.37,10.39,77.66
5,6,91.54,28.68,60.58,7.63,82.36
6,7,82.65,21.10,62.79,2.50,65.30
7,8,75.53,28.87,59.05,6.83,71.47
8,9,81.14,43.63,76.72,21.54,81.59
9,10,74.25,47.39,79.87,24.31,75.86


### Behavioural

,user,behav_vm_mean,behav_steps_per_hour,behav_frac_standing,behav_frac_lying,behav_mvpa_min,behav_screen_min,behav_caffeine_n
0,1,56.02,956.50,0.49,0.13,140.0,58.0,9
1,2,46.80,914.59,0.48,0.13,52.0,32.0,0
2,3,56.72,1331.38,0.52,0.08,100.0,15.0,0
3,4,47.87,846.40,0.44,0.13,110.0,130.0,0
4,5,41.31,704.78,0.35,0.16,110.0,10.0,0
5,6,50.82,915.24,0.47,0.16,110.0,176.0,0
6,7,45.72,907.37,0.47,0.22,180.0,120.0,0
7,8,44.98,1285.77,0.44,0.19,140.0,80.0,0
8,9,57.86,1012.78,0.45,0.11,180.0,90.0,0
9,10,55.25,1033.61,0.48,0.16,90.0,15.0,4


### Psychological

,user,psych_meq,psych_stai_state,psych_stai_trait,psych_daily_stress,psych_bis,psych_panas_pos_day,psych_panas_neg_day
0,1,47.0,41.0,43.0,23.0,22.0,17.00,11.75
1,2,52.0,24.0,39.0,26.0,26.0,30.00,13.50
2,3,59.0,27.0,27.0,11.0,19.0,32.00,11.50
3,4,60.0,28.0,40.0,10.0,20.0,24.50,13.25
4,5,52.0,54.0,47.0,41.0,25.0,28.25,18.75
5,6,48.0,32.0,47.0,41.0,19.0,26.75,13.25
6,7,46.0,33.0,NaN,69.0,19.0,14.75,13.50
7,8,64.0,32.0,39.0,74.0,24.0,28.00,11.75
8,9,44.0,36.0,41.0,38.0,20.0,30.00,12.75
9,10,38.0,39.0,46.0,14.0,19.0,22.50,22.00


## Targets

In [4]:
out['features'][['user'] + TARGETS].round(2).describe()

,user,sleep_efficiency,tst_min,waso_min,sleep_fragmentation_index,psqi
count,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000
mean,11.523810,83.648095,327.904762,62.666667,24.277619,5.238095
std,6.652962,6.759131,71.792691,28.728615,11.478506,1.997618
min,1.000000,73.490000,236.000000,21.000000,6.730000,2.000000
25%,6.000000,76.470000,302.000000,46.000000,15.400000,4.000000
50%,12.000000,84.920000,336.000000,53.000000,23.130000,5.000000
75%,17.000000,89.390000,343.000000,78.000000,31.610000,7.000000
max,22.000000,94.230000,578.000000,118.000000,45.530000,9.000000
